# Designing Networks

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TheRedElement/SNNIB/blob/main/tutorials/designing_networks_tutorial.ipynb)
If you run this notebook in [google colab](https://colab.google/), you should not need to install anything else.
You might, however, need to restart your session to have the required versions of various packages ready.

In [5]:
#in case you are in colab
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip3 install git+https://github.com/TheRedElement/SNNIB.git

In [33]:
#%%imports
import importlib

import json
import numpy as np
from plotly.subplots import make_subplots

from snnib import scaling

## Designing your custom network
* you can easily design custom networks by creating a `SNNIB` compatible file
    * see [schema](https://github.com/TheRedElement/snnib/raw/refs/heads/main/data/schema.json) for a description of required information

In [ ]:
n_neurons = 20
t_sim = 120
dt = 1
steps = t_sim//dt   #need to be int

#define metadata
meta = dict(
    t_sim=t_sim,
    t_sim_unit="",
    dt=dt,
    dt_unit="",
    steps=steps,
)

#generate neurons
rng = np.random.default_rng()
neurons_coords = rng.uniform(0, 1, (n_neurons,3))
sortidx = np.argsort(np.sum(neurons_coords**2, axis=1))
neurons_coords = neurons_coords[sortidx,:]

##define spike-trains (sweep outgoing from `spike_root`)
spike_root = np.array([0,0,0])
n_spikes_corr = 2   #number of correlated spikes
n_spikes_rand = 0   #maximum number of random spikes
# distances = scaling.minmaxscale(np.sqrt(np.sum((neurons_coords - spike_root)**2, axis=1)), xmin_ref=0, xmax_ref=np.sqrt(3))
distances = scaling.minmaxscale(np.sqrt(np.sum((neurons_coords[:,:2] - spike_root[:2])**2, axis=1)), xmin_ref=0, xmax_ref=np.sqrt(3))

# spiketrains = np.array([  #based on neuron position
#     (steps * (i+1) * distances // n_spikes_corr).astype(int)
# for i in range(n_spikes_corr)]).transpose(1,0).tolist()
spiketrains = ( #based on neuron index
    np.linspace(0, steps, n_neurons*n_spikes_corr)
        .reshape(n_spikes_corr,-1)
        .transpose(1,0)
        .astype(int)
        .tolist()
)
# print(spiketrains)
    

for i in range(len(spiketrains)):
    spiketrains[i] += set(rng.integers(0, steps, n_spikes_rand).tolist())            #random spikes

##combine
neurons = [neurons_coords[i].tolist() + [spiketrains[i]] for i in range(len(neurons_coords))]
# print(neurons)

#generate synapses
p_synapse = 0.1 #probability that a synapse exists
synapses = np.array([[pre, post, float(rng.uniform(0,1))] for pre in range(n_neurons) for post in range(n_neurons)  if pre != post])
synapses = synapses[(synapses[:,2] > (1-p_synapse))].tolist()
# print(synapses)



In [ ]:
#save generated network
snnib_json = dict(meta=meta, neurons=neurons, synapses=synapses)    #snnib uses json to avoid additional dependencies in the blender add-on
with open("./temp_custom_net.json", "w") as f:
    json.dump(snnib_json, f)

In [ ]:
#testplot
fig = make_subplots(1,1)
fig.add_trace(dict(
    x=neurons_coords[:,0],
    y=neurons_coords[:,1],
    z=neurons_coords[:,2],
    type="scatter3d",
    mode="markers",
    showlegend=False,
    marker=dict(
        color=[n[-1][0] for n in neurons],
        showscale=True,
    ),
))
fig.add_traces([dict(
    x=[neurons_coords[int(pre),0],neurons_coords[int(post),0]],
    y=[neurons_coords[int(pre),1],neurons_coords[int(post),1]],
    z=[neurons_coords[int(pre),2],neurons_coords[int(post),2]],
    type="scatter3d",
    mode="lines",
    showlegend=False,
    line=dict(
        color="#000000",
    ),
) for (pre, post, _) in synapses])
fig.show()

